# Enrichment cutoff playground

Change the knobs in the first code cell, then run the cells below it. Nothing is written to disk and nothing is registered in lamindb.

- Changing `NEIGHBORS` or `SPARSE_FRACTION` changes which cells count as lonely, so rerun from the "Lonely cells" cell down.
- Changing anything else (`STEP_WIDTH`, `MIN_CELLS_PER_STEP`, `ALPHA`, `EFFECT_SIZE_CEILING`, `QUIET_STEPS_REQUIRED`, `CANDIDATE_MAX`) only needs the "Rule" cell and the cells after it.
- Set `RUN_LABEL` before each rerun. The "Compare runs" cell lines up the cutoffs from every labelled run in this session.
- To use the commercial samples, point `COHORT_DIR` at `.../xenium/commercial` and list its sample names (`TIS04788`, `TIS04789`, `TIS05392`, `TIS05393`, `TIS05394`, `TIS05397_008`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy.stats import binom, poisson
from sklearn.neighbors import NearestNeighbors

## Knobs

In [ ]:
COHORT_DIR   = '/home/workspace/PCD_data/xenium/fh1_pretreatment'
SAMPLE_NAMES = [
    'TIS04399-001',
    'TIS02434-001',
    'TIS02437-001',
    'TIS04398-001',
    'TIS02611-001',
    'TIS02603-001',
    'TIS02604-001',
    'TIS02613-001',
    'TIS02421-001',
    'TIS02430-001',
    'TIS02424-001',
    'TIS02439-001',
    'TIS02594-001',
]

NEIGHBORS            = 10
SPARSE_FRACTION      = 0.10
STEP_WIDTH           = 10
MIN_CELLS_PER_STEP   = 30
QUIET_STEPS_REQUIRED = 1
ALPHA                = 0.05
EFFECT_SIZE_CEILING  = None
CANDIDATE_MAX        = 150

NOISE_QUANTILE = 0.999
RUN_LABEL      = 'default'

`EFFECT_SIZE_CEILING = None` stops at the first step that is not significantly lonelier than chance (`ALPHA`). Set it to a share such as `0.13` to stop at the first step whose lonely share is at or below that ceiling instead, and `ALPHA` is then ignored.

`QUIET_STEPS_REQUIRED = 1` stops at the first quiet step. Set it to 2 or 3 to require that many quiet steps in a row, so one lucky step does not end the search. The cutoff is then the start of the first run of quiet steps.

## Load samples

In [ ]:
counts       = {}
coordinates  = {}
noise_floor  = {}
for name in SAMPLE_NAMES:
    adata = ad.read_h5ad(f'{COHORT_DIR}/data/samples/{name}.h5ad')
    counts[name]      = np.asarray(adata.X.sum(axis=1)).ravel()
    coordinates[name] = adata.obsm['spatial']
    metrics = pd.read_csv(f'{COHORT_DIR}/data/outs/{name}/metrics_summary.csv')
    noise_floor[name] = poisson.ppf(NOISE_QUANTILE, metrics.loc[0, 'estimated_number_of_false_positive_transcripts_per_cell']) + 1

runs = {}

## Lonely cells

The cells whose distance to their `NEIGHBORS`-th nearest neighbor is in the top `SPARSE_FRACTION`.

In [ ]:
in_sparse_tissue = {}
for name in SAMPLE_NAMES:
    neighbor_distance = NearestNeighbors(n_neighbors=NEIGHBORS + 1).fit(coordinates[name]).kneighbors(coordinates[name])[0][:, -1]
    in_sparse_tissue[name] = neighbor_distance > np.quantile(neighbor_distance, 1 - SPARSE_FRACTION)

## Rule

Counts are grouped in steps of `STEP_WIDTH`. A step is quiet when it has at least `MIN_CELLS_PER_STEP` cells and is not lonelier than the rule allows. The cutoff is the start of the first run of `QUIET_STEPS_REQUIRED` quiet steps in a row, floored at the noise floor. NaN means no such run exists up to `CANDIDATE_MAX`.

In [ ]:
step_rows        = []
cutoffs          = {}
removed_fraction = {}
for name in SAMPLE_NAMES:
    for step_start in range(0, CANDIDATE_MAX, STEP_WIDTH):
        in_step        = (counts[name] >= step_start) & (counts[name] < step_start + STEP_WIDTH)
        cells_in_step  = in_step.sum()
        lonely_in_step = (in_step & in_sparse_tissue[name]).sum()
        lonely_share   = lonely_in_step / cells_in_step if cells_in_step else np.nan
        p_value        = binom.sf(lonely_in_step - 1, cells_in_step, SPARSE_FRACTION)

        is_quiet = (p_value >= ALPHA) if EFFECT_SIZE_CEILING is None else (lonely_share <= EFFECT_SIZE_CEILING)
        is_quiet = is_quiet and cells_in_step >= MIN_CELLS_PER_STEP
        step_rows.append((name, step_start, cells_in_step, lonely_in_step, lonely_share, p_value, is_quiet))

    sample_steps  = [row for row in step_rows if row[0] == name]
    quiet_flags   = np.array([row[6] for row in sample_steps], dtype=int)
    run_complete  = np.convolve(quiet_flags, np.ones(QUIET_STEPS_REQUIRED, dtype=int), mode='valid') == QUIET_STEPS_REQUIRED
    cutoffs[name] = max(sample_steps[run_complete.argmax()][1], noise_floor[name]) if run_complete.any() else np.nan
    removed_fraction[name] = (counts[name] < cutoffs[name]).mean() if not np.isnan(cutoffs[name]) else np.nan

step_table = pd.DataFrame(step_rows, columns=['sample', 'step_start', 'cells', 'lonely', 'lonely_share', 'p_value', 'quiet'])
runs[RUN_LABEL] = pd.Series(cutoffs)

pd.DataFrame({'noise_floor': noise_floor, 'enrichment_cutoff': cutoffs, 'fraction_removed': removed_fraction})

## Compare runs

Cutoffs from every `RUN_LABEL` run in this session. Rerunning the rule cell with the same label replaces that column.

In [ ]:
pd.DataFrame(runs)

## Plots

Left: the share of lonely cells in each step. Blue bars are steps that are still lonelier than the rule allows, grey bars are quiet steps, and pale bars have too few cells to judge. Right: the cumulative enrichment curve from the sweep plot, with the fraction of cells removed in orange. Purple dash-dot is the cutoff, green dotted is the noise floor. With `QUIET_STEPS_REQUIRED` above 1, a grey step that is not part of a long enough run of grey steps is skipped by the cutoff.

In [ ]:
candidate_min_counts = np.arange(1, CANDIDATE_MAX + 1)

fig, axes = plt.subplots(len(SAMPLE_NAMES), 2, figsize=(14, 4 * len(SAMPLE_NAMES)), squeeze=False)
for row_axes, name in zip(axes, SAMPLE_NAMES):
    steps = step_table[step_table['sample'] == name]
    bar_colors = ['lightgray' if cells < MIN_CELLS_PER_STEP else 'gray' if quiet else 'tab:blue'
                  for cells, quiet in zip(steps['cells'], steps['quiet'])]

    step_ax, curve_ax = row_axes
    step_ax.bar(steps['step_start'], steps['lonely_share'].fillna(0), width=STEP_WIDTH, align='edge', color=bar_colors, edgecolor='white')
    step_ax.axhline(SPARSE_FRACTION, color='black', ls='--', label=f'expected by chance ({SPARSE_FRACTION:.0%})')
    if EFFECT_SIZE_CEILING is not None:
        step_ax.axhline(EFFECT_SIZE_CEILING, color='tab:red', ls=':', label=f'ceiling ({EFFECT_SIZE_CEILING:.0%})')
    step_ax.set_xlabel('total counts (start of step)')
    step_ax.set_ylabel('share of cells that are lonely')
    step_ax.set_title(name)

    removed = counts[name][:, None] < candidate_min_counts[None, :]
    removed_per_candidate = np.maximum(removed.sum(axis=0), 1)
    enrichment = (removed & in_sparse_tissue[name][:, None]).sum(axis=0) / removed_per_candidate / SPARSE_FRACTION
    curve_ax.plot(candidate_min_counts, enrichment, color='tab:blue')
    curve_ax.axhline(1, color='gray', ls=':')
    curve_ax.set_xlabel('minimum counts')
    curve_ax.set_ylabel('enrichment in sparse tissue (fold)', color='tab:blue')
    curve_ax.set_title(name)
    fraction_axis = curve_ax.twinx()
    fraction_axis.plot(candidate_min_counts, removed.mean(axis=0), color='tab:orange')
    fraction_axis.set_ylabel('fraction of cells removed', color='tab:orange')

    for ax in row_axes:
        ax.axvline(noise_floor[name], color='tab:green', ls=':', label=f'noise floor ({noise_floor[name]:.0f})')
        if not np.isnan(cutoffs[name]):
            ax.axvline(cutoffs[name], color='tab:purple', ls='-.', label=f'cutoff ({cutoffs[name]:.0f})')
        ax.legend(fontsize=8, loc='upper right')

fig.tight_layout()
plt.show()